# Senate Profile LLM Extraction Pipeline
**DSBA 6010 — Chloe Partridge**

Aligned with Liu et al. (USENIX Security 2025) *Evaluating LLM-based Personal Information Extraction and Countermeasures*.

Key features:
- **Prompt-style ablation** — direct, pseudocode, ICL (Section 4.2 / Table 13)
- **Religion signal annotation** — LLM-based pre-classification of bio text as `explicit` / `not_explicit` (Section 8b)
- **Traditional baselines** — keyword search, regex, spaCy NER (Tables 4–5)
- **Evaluation metrics** — Accuracy, Rouge-1, BERT score with stratified religion analysis (Section 6.1.4)
- **Model comparison** — 8B vs 70B (Table 3 / Section 6.2)

In [1]:
# Install required packages for all supported providers
# !pip install beautifulsoup4 google-generativeai openai groq pandas tqdm rouge-score bert-score spacy
# !python -m spacy download en_core_web_sm

## 2. Configuration & Session Setup

###  API Configuration & Session tracking
Initializes API client, sets temperature and token limits, and prepares session metadata.

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import json, time, re, random, os, datetime
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# ════════════════════════════════════════════════════════════════════════════
# CENTRALIZED SESSION INITIALIZATION
# ════════════════════════════════════════════════════════════════════════════
# 
# The modules package now provides factory functions that initialize the entire
# session with one call. This replaces ~80 lines of configuration boilerplate.
#
# What initialize_pipeline_session() does:
#   1. Loads model configuration from JSON (groq_config_extraction.json)
#   2. Sets up API client (Groq with GROQ_API_KEY env var or config file)
#   3. Initializes spaCy NER model (en_core_web_sm)
#   4. Collects HTML files from external_data/senate_html/
#   5. Creates PipelineConfig dataclass with all settings
#
# Returns a dictionary with all session state ready to use.
# ════════════════════════════════════════════════════════════════════════════

from modules import initialize_pipeline_session

# Initialize the session (loads config, API client, spaCy model, HTML files)
session = initialize_pipeline_session(
    config_path="../configs/model_configs/groq_config_extraction.json",
    prompt_styles=["direct", "pseudocode", "icl"]
)

# Unpack session for convenient access
session_config = session['session_config']
html_files = session['html_files']
nlp = session['nlp']
OUTPUT_DIR = session['output_dir']
HTML_DIR = session['html_dir']

# ── Initialize ablation subset with fixed seed for reproducibility ────────
# Fixed seed=42 ensures same 25 senators across runs
import random as py_random
py_random.seed(42)

# Select 25 senators for ablation study
ablation_senators = py_random.sample(
    [f.stem for f in html_files],
    min(25, len(html_files))
)

print(f"✓ Model: {session_config.model} | {len(html_files)} HTML files")
print(f"✓ Prompt styles: {', '.join(session_config.prompt_styles)}")
print(f"✓ Output dir: {OUTPUT_DIR}")
print(f"✓ Ablation subset: {len(ablation_senators)} senators (seed=42)")

/Users/chloe/LLM-Based-Personal-Profile-Extraction/pie_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ Model: mixtral-8x7b-32768 | 100 HTML files
✓ Prompt styles: direct, pseudocode, icl
✓ Output dir: /Users/chloe/LLM-Based-Personal-Profile-Extraction/experiments/../outputs/senate_results
✓ Ablation subset: 25 senators (seed=42)


## 5. API Infrastructure

Helper functions for LLM API calls with automatic retry on rate-limit errors.
Includes exponential backoff, JSON parsing, and model-agnostic abstraction layer.

In [4]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 6: LLM EXTRACTION — Task 1 (Full Name, Birthdate, Gender, Education, etc.)
# ════════════════════════════════════════════════════════════════════════════
#
# This section runs the main LLM extraction pipeline on all senate HTML profiles.
#
# Key features:
#   - Uses Groq API with configured model (8B or 70B)
#   - Tests three prompt engineering strategies: direct, pseudocode, ICL
#   - Includes rate limiting and retry logic
#   - Resume-safe: stops/restarts without re-processing completed senators
#   - Results cached to JSON: results_raw.json
#
# Approx runtime: 2-5 minutes depending on model + number of styles
# ════════════════════════════════════════════════════════════════════════════

from modules import run_main_pipeline

# Execute the main pipeline with resume capability
# Returns: dict with "results", "output_path", "done_count", "total_count"
pipeline_result = run_main_pipeline(
    html_files=html_files,
    session_config=session_config,
    resume=True  # Skip already-processed senators
)

print(f"\n✓ Pipeline complete: {pipeline_result['done_count']}/{pipeline_result['total_count']} senators processed")

✓ Resuming: 100 senators already processed
Senators remaining: 0/100
Rate limit: 6s between calls
Styles: direct, pseudocode, icl



Processing senators: 0it [00:00, ?it/s]


✓ Pipeline complete: 100 senators processed

✓ Pipeline complete: 100/100 senators processed


In [5]:
# Convert raw JSON results to CSV format for easier analysis
# (This is just data formatting from run_main_pipeline output)

from modules import T1_FIELDS

with open(OUTPUT_DIR / "results_raw.json") as f:
    raw_results = json.load(f)

# Flatten Task 1 results to CSV
task1_rows = []
for r in raw_results:
    t1_data = r.get("task1_pii", {})
    prompt_style = r.get("prompt_style", "unknown")
    
    # Handle both single-style and all-styles results
    if prompt_style == "all_styles":
        # All three styles were run — create separate rows for each style
        for style_name, style_result in t1_data.items():
            row = {
                "senator_id": r["senator_id"],
                "prompt_style": style_name,
                "extraction_error": style_result.get("error")
            }
            for field in T1_FIELDS:
                row[field] = style_result.get(field)
            task1_rows.append(row)
    else:
        # Single style was run
        row = {
            "senator_id": r["senator_id"],
            "prompt_style": prompt_style,
            "extraction_error": t1_data.get("error")
        }
        for field in T1_FIELDS:
            row[field] = t1_data.get(field)
        task1_rows.append(row)

df_pred = pd.DataFrame(task1_rows)
df_pred.to_csv(OUTPUT_DIR / "task1_pii.csv", index=False)
print(f"✓ Saved {len(df_pred)} rows to task1_pii.csv")
print(f"\nPrompt styles processed:")
print(df_pred["prompt_style"].value_counts().to_string())

✓ Saved 300 rows to task1_pii.csv

Prompt styles processed:
prompt_style
direct        100
pseudocode    100
icl           100


In [6]:
# Display coverage statistics across all extracted fields
print("Field Coverage Statistics (LLM Extraction Results)")
print("=" * 60)
for col in ["full_name", "birthdate", "gender", "education", "committee_roles", "religious_affiliation"]:
    if col in df_pred.columns:
        filled = df_pred[col].notna().sum()
        pct = 100 * filled / len(df_pred)
        print(f"  {col:30s}: {filled:3d}/{len(df_pred)} ({pct:5.1f}%)")

Field Coverage Statistics (LLM Extraction Results)
  full_name                     : 300/300 (100.0%)
  birthdate                     :  51/300 ( 17.0%)
  gender                        : 267/300 ( 89.0%)
  education                     : 300/300 (100.0%)
  committee_roles               : 299/300 ( 99.7%)
  religious_affiliation         :  97/300 ( 32.3%)


## 8. Baseline Methods (Liu et al. Section 5, Tables 4–5)

Implements traditional information extraction methods for comparison:
- **Regex baseline** — Rule-based patterns for name, email, phone
- **spaCy NER baseline** — Pre-trained neural entity recognition for PERSON, ORG, GEO

Purpose: Quantify LLM superiority on canonical PII extraction tasks. 
This section tests both methods on the same HTML data then compares coverage vs. LLM accuracy.

In [7]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 8: BASELINE METHODS — Regex + spaCy NER (Liu et al. Tables 4–5)
# ════════════════════════════════════════════════════════════════════════════
#
# For comparison baseline, we also extract PII using traditional pattern matching:
#   - Regex baseline: Hand-crafted rules for common name/email/phone patterns
#   - spaCy NER: Pre-trained neural network (PERSON, ORG, GPE entities)
#
# This allows us to quantify LLM superiority on canonical PII extraction tasks.
# Approx runtime: 30-60 seconds (fast — no API calls)
# ════════════════════════════════════════════════════════════════════════════

In [8]:
# Execute baseline extraction methods on all HTML files
# run_baselines() handles regex + spaCy NER on each profile
# and returns coverage statistics comparing both methods

from modules import run_baselines, REGEX_PATTERNS

baseline_metrics = run_baselines(
    html_files=html_files,
    nlp=nlp,
    regex_patterns=REGEX_PATTERNS,
    output_dir=OUTPUT_DIR
)

print(f"✓ Baseline extraction complete")
print(f"  Results saved to: {OUTPUT_DIR / 'baselines.csv'}")


Running baselines on 100 profiles...


Baselines:   0%|          | 0/100 [00:00<?, ?it/s]

✓ Saved baseline results to /Users/chloe/LLM-Based-Personal-Profile-Extraction/experiments/../outputs/senate_results/baselines.csv

✓ Baseline extraction complete
  Results saved to: /Users/chloe/LLM-Based-Personal-Profile-Extraction/experiments/../outputs/senate_results/baselines.csv


In [9]:
# ── Load and verify ground truth ─────────────────────────────────────────
# Ground truth CSV is generated by wiki/ballotpedia scraping + manual verification
# (Pew religion merging happens in backend during ground truth scraping phase)

OUTPUT_PATH = Path("../external_data/ground_truth/senate_ground_truth.csv")

if not OUTPUT_PATH.exists():
    print(f"⚠ Ground truth file not found: {OUTPUT_PATH}")
    print("  This file should be pre-generated from senate_llm_pipeline_V1.ipynb (ground truth scraping)")
    print("  Evaluation will not work without it.")
    df_gt = None
else:
    df_gt = pd.read_csv(OUTPUT_PATH)
    print("Ground Truth Summary:")
    print(f"  Total senators: {len(df_gt)}")
    print(f"  Fields available: {len(df_gt.columns)}")
    print(f"  Empty fields per senator (avg): {df_gt.isna().sum().mean():.1f}")
    
    # Show field coverage
    print("\nField Coverage (Ground Truth):")
    for col in ["name", "birthdate", "gender", "religion", "committee_roles"]:
        if col in df_gt.columns:
            covered = df_gt[col].notna().sum()
            pct = 100 * covered / len(df_gt)
            print(f"  {col:20s}: {covered:3d}/{len(df_gt)} ({pct:5.1f}%)")
    
    print(f"\n✓ Ground truth ready for evaluation")

Ground Truth Summary:
  Total senators: 99
  Fields available: 11
  Empty fields per senator (avg): 12.2

Field Coverage (Ground Truth):
  name                :  99/99 (100.0%)
  birthdate           :  94/99 ( 94.9%)
  gender              :  94/99 ( 94.9%)
  religion            :  96/99 ( 97.0%)
  committee_roles     :  79/99 ( 79.8%)

✓ Ground truth ready for evaluation


In [10]:
# ════════════════════════════════════════════════════════════════════════════
# SECTION 9: COMPREHENSIVE EVALUATION (With Result Caching)
# ════════════════════════════════════════════════════════════════════════════
#
# Compare LLM predictions vs ground truth using structured evaluation.
#
# Features:
#   - Automatic result caching (evaluate_all_styles checks cache before recomputing)
#   - Stratified metrics: accuracy, ROUGE-1, BERT score, hierarchical religion match
#   - Component-level breakdown for education (degree/institution/year matching)
#   - Supports fuzzy name matching fallback for senator_id mismatches
#
# Runtime: 30-90 seconds (much faster on second run due to caching)
# ════════════════════════════════════════════════════════════════════════════

from modules import (
    load_and_merge_results,
    evaluate_all_styles,
    print_evaluation_summary,
)

# Load and merge all predictions with ground truth
print("Loading predictions and ground truth...")
merged_data = load_and_merge_results(
    pred_path=OUTPUT_DIR / "task1_pii.csv",
    gt_path=Path("../external_data/ground_truth/senate_ground_truth.csv"),
    merge_method="exact"
)

# Compute evaluation metrics with caching
# (Second run on same data will load cached results instead of recomputing)
print("\nComputing evaluation metrics...")
eval_results = evaluate_all_styles(
    merged_by_style=merged_data['merged_by_style'],
    output_dir=OUTPUT_DIR,
    overwrite=False  # Set to True to recompute even if cache exists
)

# Print formatted summary
print_evaluation_summary(eval_results)

Loading predictions and ground truth...
✓ Loaded 300 predictions
✓ Loaded 99 ground truth records
✓ Prompt styles: ['direct' 'pseudocode' 'icl']


Computing evaluation metrics...
Evaluating [DIRECT] (99 records)


AttributeError: 'str' object has no attribute 'get'

## 10. Prompt Ablation Study (Liu et al. Section 4.2, Table 13)

Rigorous comparison of all three prompt styles on a **fixed held-out subset of 25 senators**.
Each senator is extracted independently using direct, pseudocode, and ICL prompts.

**Purpose:** Quantify which fields benefit most from structured prompts (e.g., pseudocode) vs. demonstrations (e.g., ICL).
This replicates Liu et al.'s ablation methodology (Section 4.2, Table 13).

**Reproducibility:**
- Fixed random seed (42) ensures same 25 senators across runs
- Separate results file (`ablation_results.json`) keeps ablation independent from main pipeline
- Resume-safe — skips already-completed combinations
- 3-second rate limit between API calls to respect quota

**Expected outcome:** ICL shows largest gains on education, committee roles, and religious affiliation (Liu et al. reports ~15-25% improvement)

In [ ]:
# ── Define all prompt styles for ablation ───────────────────────────────
# These are the same prompts as the main pipeline, but applied systematically
# to a fixed subset for controlled comparison
ABLATION_STYLES = {
    "direct": TASK1_DIRECT,
    "pseudocode": TASK1_PSEUDOCODE,
    "icl": TASK1_ICL
}

# ── Load or initialize ablation results ──────────────────────────────────
# Resume-safe: if ablation was interrupted, load progress and continue
ablation_path = OUTPUT_DIR / "ablation_results.json"

if ablation_path.exists():
    with open(ablation_path) as f:
        ablation_results = json.load(f)
    # Track which (senator_id, style) combos are already completed
    completed = set()
    for sid, styles_dict in ablation_results.items():
        for style in styles_dict.keys():
            completed.add((sid, style))
    print(f"✓ Resuming — {len(completed)} senator-style combinations already processed")
else:
    ablation_results = {}
    completed = set()

# ── Compute remaining tasks ──────────────────────────────────────────────
# Each senator needs to be extracted with all 3 styles
ablation_tasks = [(sid, style) for sid in ablation_senators for style in ABLATION_STYLES.keys()]
remaining_tasks = [t for t in ablation_tasks if t not in completed]

print(f"Remaining ablation tasks: {len(remaining_tasks)}/{len(ablation_tasks)}")
print()

# Main ablation loop: extract each senator with each prompt style
for senator_id, style_name in tqdm(remaining_tasks, desc="Ablation loop"):
    # Load HTML and extract clean text (same preprocessing as main pipeline)
    html_file = [f for f in html_files if f.stem == senator_id]
    if not html_file:
        continue
    
    # Read and clean HTML
    html = html_file[0].read_text(encoding="utf-8", errors="ignore")
    text = extract_readable_text(html)
    
    # Extract with current style using Groq API
    prompt = ABLATION_STYLES[style_name]
    result = call_groq(prompt, text)
    
    # Store result in nested dict: {senator_id: {style_name: result, ...}, ...}
    if senator_id not in ablation_results:
        ablation_results[senator_id] = {}
    
    ablation_results[senator_id][style_name] = result
    
    # Save incrementally (critical for resume safety and data preservation)
    with open(ablation_path, "w") as f:
        json.dump(ablation_results, f, indent=2)
    
    # Rate limit protection — 3 seconds between API calls to avoid quota errors
    time.sleep(3)

print(f"\n✓ Ablation complete. Results saved to: ablation_results.json")

In [ ]:
# ── Coverage comparison: field coverage per prompt style ─────────────────
print("\n" + "=" * 80)
print(" PROMPT ABLATION — FIELD COVERAGE COMPARISON")
print("=" * 80)
print(f"\nAblation subset size: {len(ablation_senators)} senators")
print(f"Ablation styles: {', '.join(ABLATION_STYLES.keys())}")
print()

# Calculate coverage for each field by prompt style
coverage_by_style = {}
for style in ABLATION_STYLES.keys():
    style_data = df_ablation[df_ablation["prompt_style"] == style].copy()

    error_mask = style_data.get("extraction_error", pd.Series(dtype=str)).notna()
    valid_data = style_data[~error_mask]
    n_errors = int(error_mask.sum())
    n_valid = len(valid_data)

    if n_valid == 0:
        print(f"  ⚠ {style}: all {n_errors} results are errors — delete ablation_results.json and rerun")
        continue

    coverage_by_style[style] = {}
    for field in T1_FIELDS_ABLATION:
        filled = int(valid_data[field].notna().sum())
        pct = (filled / n_valid * 100)
        coverage_by_style[style][field] = {"filled": filled, "total": n_valid, "pct": pct}

if not coverage_by_style:
    print("⚠ No valid results to display.")
else:
    # Print coverage table
    print(f"{'Field':<35} | {'DIRECT':^15} | {'PSEUDOCODE':^15} | {'ICL':^15}")
    print("-" * 80)
    for field in T1_FIELDS_ABLATION:
        row_str = f"{field:<35} | "
        for style in ["direct", "pseudocode", "icl"]:
            if style in coverage_by_style:
                c = coverage_by_style[style][field]
                row_str += f"{c['filled']}/{c['total']} ({c['pct']:5.1f}%) | "
            else:
                row_str += f"{'N/A':^15} | "
        print(row_str)
    print("=" * 80)

    # Show average coverage per style
    print("\n OVERALL COVERAGE BY PROMPT STYLE:")
    for style in ["direct", "pseudocode", "icl"]:
        if style in coverage_by_style:
            avg_cov = sum(coverage_by_style[style][f]["pct"] for f in T1_FIELDS_ABLATION) / len(T1_FIELDS_ABLATION)
            print(f"  {style:<15}: {avg_cov:6.1f}% (avg across all fields)")
        else:
            print(f"  {style:<15}: N/A (all errors)")

    print("\n KEY OBSERVATIONS:")
    print("  • Compare direct vs pseudocode: Liu et al. find pseudocode slightly better for structured fields")
    print("  • Compare direct/pseudocode vs ICL: Liu et al. find ICL best for occupation-type fields")
    print("    (committee_roles, education, religious_affiliation)")
    print("=" * 80 + "\n")

## 11. Appendix: Religion Taxonomy & Hierarchical Matching

To give partial credit for related religions (e.g., Methodist → Christian),
we use a hierarchical taxonomy. Exact matches score 1.0, parent-child matches
(e.g., Methodist vs Christian) score 0.7, and unrelated religions score 0.0.

This better reflects partial correctness: extracting "Christian Methodist" when
GT is "Methodist" is partially correct even if not exact.

### Helper Functions for Evaluation

```python
# Religion hierarchy: maps denominations to their parent category
RELIGION_HIERARCHY = {
    # Catholic tradition
    "catholic": "catholic",
    "roman catholic": "catholic",
    
    # Protestant denominations
    "methodist": "protestant",
    "united methodist": "protestant",
    "methodist church": "protestant",
    "baptist": "protestant",
    "southern baptist": "protestant",
    "presbyterian": "protestant",
    "episcopal": "protestant",
    "episcopalian": "protestant",
    "lutheran": "protestant",
    "evangelical": "protestant",
    "pentecostal": "protestant",
    "assemblies of god": "protestant",
    
    # Broadly Christian (covers generic "Christian" answers)
    "christian": "christian",
    "christian (unspecified)": "christian",
    "protestant": "protestant",
    
    # Other major religions
    "jewish": "jewish",
    "judaism": "jewish",
    "muslim": "muslim",
    "islam": "muslim",
    "jewish orthodox": "jewish",
    "orthodox": "orthodox",
    "mormon": "mormon",
    "church of jesus christ": "mormon",
    "lds": "mormon",
    "unitarian": "unitarian",
    "unitarian universalist": "unitarian",
    
    # Secular / None
    "atheist": "none",
    "agnostic": "none",
    "none": "none",
    "no religion": "none",
    "unaffiliated": "none",
}

def get_religion_category(religion_str):
    """
    Normalize religion string and return its category from the hierarchy.
    """
    if pd.isna(religion_str) or str(religion_str).strip() == "":
        return None
    
    norm = str(religion_str).strip().lower()
    
    # Direct lookup
    if norm in RELIGION_HIERARCHY:
        return RELIGION_HIERARCHY[norm]
    
    # Substring matching for common phrases
    for key, category in RELIGION_HIERARCHY.items():
        if key in norm or norm in key:
            return category
    
    # Default: treat as its own category
    return norm

def religion_match_score(gt_val, pred_val):
    """
    Hierarchical religion matching:
      - 1.0 for exact match (after lowercasing)
      - 0.7 for parent-child match (e.g., Methodist vs Christian)
      - 0.0 for unrelated religions
      - NaN when either GT or pred is missing (do not score as incorrect)
    
    This allows the model to get partial credit when extracting a parent
    category (Christian) when GT is a specific denomination (Methodist),
    reflecting the fact that it's partially correct.
    """
    if pd.isna(gt_val) or str(gt_val).strip() == "":
        return float("nan")
    if pd.isna(pred_val) or str(pred_val).strip() == "":
        return float("nan")
    
    gt_norm = str(gt_val).strip().lower()
    pred_norm = str(pred_val).strip().lower()
    
    # Exact match
    if gt_norm == pred_norm:
        return 1.0
    
    # Hierarchical match
    gt_cat = get_religion_category(gt_norm)
    pred_cat = get_religion_category(pred_norm)
    
    if gt_cat and pred_cat:
        if gt_cat == pred_cat:
            # Same category but different names —  partial credit
            return 0.7
    
    # No match
    return 0.0
```

**Scoring Details:**
- **1.0** — Exact match (Methodist = Methodist)
- **0.7** — Same religious category but different names (Methodist vs Christian, Baptist vs Protestant)
- **0.0** — Different religions (Methodist vs Catholic, Christian vs Muslim)
- **NaN** — Either value missing (not scored)

This approach rewards the model for extracting the correct parent category
when it can't pin down the exact denomination, which is semantically more
reasonable than treating all non-exact matches as completely wrong.